# CS781 Capstone – Phase 4: Court Judgment Prediction

Name: Aditya Kumar | Roll No: 240059 | Email: adityaj24@iitk.ac.in

In [1]:
import os, re, gc, glob, math, json, time, random, shutil, zipfile, hashlib, subprocess, sys, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(DEVICE, NUM_GPUS)

cuda 2


In [2]:

PLAN = [

    ("hybrid",      [dict(method="full"),    dict(method="lorma"),   dict(method="kd")]),

]
EXP_TAG = "p4"
ENSEMBLE = True
ENSEMBLE_SAME_ARCH = True
MAX_PARAMS_M = 300
SMOKE = False

hf_token = os.environ.get("HF_TOKEN")
HF_REPO, P4_DIR = "Exploration-Lab/CS-781-Capstone", "Phase_4_CJPE"
LOCAL_DATA_DIR = None
INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
OUT_ROOT = os.environ.get("CS781_OUT_ROOT", "/kaggle/working")

CHUNK_WORDS = "auto"
WINDOW = 512
K_CHUNKS = 4
DOC_MODE = "tail"
FEAT_STD = True
POOL = "attn"
USE_POS_FEATURE = True
BIDIRECTIONAL = True

EPOCHS = {"full": 2, "partial": 3, "bitfit": 3, "lora": 3, "lorma": 3, "adapter": 3, "kd": 3}
LR     = {"full": 5e-5, "partial": 1e-4, "bitfit": 1e-3, "lora": 1e-3, "lorma": 5e-4, "adapter": 5e-4, "kd": 1e-4}
HEAD_LR, WEIGHT_DECAY, WARMUP_FRAC, PATIENCE = 3e-3, 0.01, 0.06, 2
BATCH_DOCS, GRAD_ACCUM = 4, 2
LABEL_SMOOTH, CLASS_BALANCE, THRESH_TUNE = 0.05, "sqrt", True
TRAIN_DOC_LIMIT = 15000
RUN_MAX_MIN = 100
PEFT = dict(r=16, alpha=32, targets=("q_proj", "k_proj", "v_proj", "out_proj"), lorma_targets=("q_proj", "v_proj"), bottleneck=64, partial_last=4)
KD_T, KD_ALPHA, STUDENT_KEEP_EVERY = 2.0, 0.5, 2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("plan:", [(a, [r["method"] for r in rs]) for a, rs in PLAN])

plan: [('hybrid', ['full', 'lorma', 'kd'])]


In [3]:
ROPE_BASE = 10000.0

# Phase-1 architecture 1: decoder-only
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class DecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_seq_len:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Phase-1 architecture 2: encoder-decoder (encoder used)
class MultiHeadAttention(nn.Module):
    """Generic multi-head attention. causal=True masks future query->key positions (self-attn
    in the decoder). query and key/value can come from different sequences (cross-attention)."""
    def __init__(self, d_model, n_heads, dropout=0.0, causal=False, max_seq_len=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.causal = causal
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if causal:
            mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
            self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x_q, x_kv):
        B, T, C = x_q.shape
        S = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if self.causal:
            scores = scores.masked_fill(~self.causal_mask[:T, :S], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

def feed_forward(d_model, d_ff, dropout):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_tgt_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=True, max_seq_len=max_tgt_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ff(self.ln3(x))
        return x

class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff,
                 max_src_len, max_tgt_len, dropout=0.1):
        super().__init__()
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("enc_pos", sinusoidal_positional_encoding(max_src_len, d_model), persistent=False)
        self.register_buffer("dec_pos", sinusoidal_positional_encoding(max_tgt_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.decoder_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, max_tgt_len, dropout) for _ in range(n_dec_layers)])
        self.enc_ln_f = nn.LayerNorm(d_model)
        self.dec_ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode(self, src_ids):
        T = src_ids.shape[1]
        x = self.drop(self.token_emb(src_ids) + self.enc_pos[:T].unsqueeze(0))
        for block in self.encoder_blocks:
            x = block(x)
        return self.enc_ln_f(x)

    def decode(self, tgt_input_ids, enc_out):
        T = tgt_input_ids.shape[1]
        x = self.drop(self.token_emb(tgt_input_ids) + self.dec_pos[:T].unsqueeze(0))
        for block in self.decoder_blocks:
            x = block(x, enc_out)
        return self.lm_head(self.dec_ln_f(x))

    def forward(self, src_ids, tgt_input_ids, tgt_target_ids=None):
        enc_out = self.encode(src_ids)
        logits = self.decode(tgt_input_ids, enc_out)
        loss = None
        if tgt_target_ids is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt_target_ids.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        enc_out = self.encode(src_ids[:, -self.max_src_len:])
        dec_ids = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            logits = self.decode(dec_ids[:, -self.max_tgt_len:], enc_out)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=src_ids.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            dec_ids = torch.cat([dec_ids, next_token], dim=1)
        return dec_ids[:, 1:]

# Phase-1 architecture 3: hybrid prefix-LM
class PrefixAwareAttention(nn.Module):
    """Causal self-attention, except query positions inside the prefix (index < prefix_len)
    also get bidirectional access to the rest of the prefix. Positions >= prefix_len are
    plain causal. A single stack, single attention type per layer - neither a fully causal
    decoder nor a two-stack encoder-decoder."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, prefix_len):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        pos = torch.arange(T, device=x.device)
        in_prefix = pos < prefix_len
        prefix_bonus = in_prefix.unsqueeze(1) & in_prefix.unsqueeze(0)
        allowed = self.causal_mask[:T, :T] | prefix_bonus

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~allowed, float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class PrefixLMBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixAwareAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x, prefix_len):
        x = x + self.attn(self.ln1(x), prefix_len)
        x = x + self.ff(self.ln2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([PrefixLMBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, prefix_len, targets=None, loss_mask=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x, prefix_len)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            per_tok = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), reduction="none").view(B, T)
            m = (torch.arange(T, device=idx.device) + 1 >= prefix_len).to(per_tok.dtype)
            loss = (per_tok * m).sum() / (m.sum() * B).clamp(min=1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, prefix_len, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond, prefix_len=min(prefix_len, idx_cond.shape[1]))
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Phase-1 architecture 4: lightweight (RoPE/RMSNorm/SwiGLU/GQA)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):

    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

class LightBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x

class LightweightLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            LightBlock(d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Phase-1 architecture 5: RoPE + RMSNorm + SwiGLU + QK-Norm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class QKNormAttention(nn.Module):
    """Full multi-head causal attention with RoPE, plus QK-Norm: Q and K are each RMSNorm'd
    per head (over the head dimension) before the dot product, bounding attention logit scale
    independent of how large Q/K activations get during training - improves stability."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.d_head)
        self.k_norm = RMSNorm(self.d_head)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

class QKBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = QKNormAttention(d_model, n_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x

class RopeSwiGLUQKNormLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([QKBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

In [4]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

def find_phase1(arch):
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_model.pt"), recursive=True)
    assert hits, f"{arch}_model.pt not found under {INPUT_ROOT}. Attach the dataset made from your Phase-1 zips."
    d = os.path.dirname(hits[0])
    for n in (f"{arch}_bpe_tokenizer.json", "bpe_tokenizer.json"):
        if os.path.exists(os.path.join(d, n)): return hits[0], os.path.join(d, n)
    t = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_bpe_tokenizer.json"), recursive=True) or glob.glob(os.path.join(INPUT_ROOT, "**", "bpe_tokenizer.json"), recursive=True)
    assert t, "tokenizer json not found next to the model"
    return hits[0], t[0]

def load_phase1_backbone(arch, path):
    ck = torch.load(path, map_location="cpu", weights_only=False); cfg, sd = ck["config"], ck["state_dict"]; g = cfg.get
    if arch == "decoder": m = DecoderLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "encdec": m = EncoderDecoderLM(g("vocab_size"), g("d_model"), g("n_enc_layers"), g("n_dec_layers"), g("n_heads"), g("d_ff"), g("max_src_len"), g("max_tgt_len"), g("dropout", 0.1))
    elif arch == "hybrid": m = PrefixLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "lightweight": m = LightweightLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("n_kv_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "rope_qknorm": m = RopeSwiGLUQKNormLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    else: raise ValueError(arch)
    m.load_state_dict(sd, strict=True)
    return m, cfg, (g("max_seq_len") or g("max_src_len"))

def block_list_name(arch): return "encoder_blocks" if arch == "encdec" else "blocks"

def strip_unused(m, arch):
    m.lm_head = nn.Identity()
    if arch == "encdec": m.decoder_blocks = nn.ModuleList(); m.dec_ln_f = nn.Identity()

def prune_layers(m, arch, keep):
    blocks = getattr(m, block_list_name(arch)); setattr(m, block_list_name(arch), nn.ModuleList([blocks[i] for i in keep]))

In [5]:

class LoRALinear(nn.Module):
    """LoRA: y = Wx + s*B(Ax). rsLoRA = same with s = alpha/sqrt(r) instead of alpha/r (rank-stabilised)."""
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x): return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """LoRMA+ : W' = (I + (alpha/r) B A) W0 (multiplicative); A=0, B=kaiming as in the authors' code, so W'=W0 at step 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.zeros(r, base.out_features)); self.B = nn.Parameter(torch.empty(base.out_features, r))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
    def forward(self, x):
        h = F.linear(x, self.base.weight); y = h + self.s * F.linear(F.linear(h, self.A), self.B)
        return y + self.base.bias if self.base.bias is not None else y

In [6]:
class BPE:
    """Phase-1 byte-level BPE (load + encode). Same merges and special ids as pretraining."""
    SPECIAL = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self):
        self.merges = {}; self.id_to_bytes = {i: bytes([i]) for i in range(256)}; self.special_to_id = {}
        for i, t in enumerate(self.SPECIAL): self.special_to_id[t] = 256 + i; self.id_to_bytes[256 + i] = t.encode()
    @staticmethod
    def _merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair: out.append(new_id); i += 2
            else: out.append(seq[i]); i += 1
        return out
    @classmethod
    def load(cls, path):
        tok = cls()
        for ps, nid in json.load(open(path, encoding="utf-8"))["merges"].items():
            a, b = (int(x) for x in ps.split(",")); tok.merges[(a, b)] = nid; tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok
    def encode_piece(self, piece):
        seq = list(piece.encode("utf-8"))
        while len(seq) > 1:
            cand = [p for p in zip(seq, seq[1:]) if p in self.merges]
            if not cand: break
            pair = min(cand, key=lambda p: self.merges[p]); seq = self._merge(seq, pair, self.merges[pair])
        return seq

class WordTok:
    def __init__(self, bpe): self.bpe, self.eos, self.pad, self.cache = bpe, bpe.special_to_id["</s>"], bpe.special_to_id["<pad>"], {}
    def encode_words(self, words, max_tok):
        ids = []
        for w in words:
            p = self.cache.get(w)
            if p is None: p = self.cache[w] = self.bpe.encode_piece(w) + [self.eos]
            ids.extend(p)
        return np.array((ids or [self.eos])[-max_tok:], dtype=np.int64)

def read_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)
def get_path(rel):
    if LOCAL_DATA_DIR: return os.path.join(LOCAL_DATA_DIR, rel)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_REPO, filename=f"{P4_DIR}/{rel}", repo_type="dataset", token=hf_token)

def parse_cases(raw, labelled):
    """[{id,text,label}] or {id: {text,label}} (label key may be label/labels/pred/outcome/judgment)."""
    pairs = list(raw.items()) if isinstance(raw, dict) else [(r.get("id", r.get("case_id")), r) for r in raw]
    out = []
    for cid, v in pairs:
        if not isinstance(v, dict): v = {"text": v}
        text = next((v[k] for k in ("text", "document", "case_text", "content") if k in v), None)
        assert text is not None, f"case {cid}: no text field (keys: {list(v)})"
        if isinstance(text, list): text = " ".join(map(str, text))
        lab = None
        if labelled:
            k = next((k for k in ("label", "labels", "pred", "outcome", "judgment", "y") if k in v), None)
            assert k is not None, f"case {cid}: no label field (keys: {list(v)})"
            lab = int(float(v[k])); assert lab in (0, 1), f"case {cid}: label {lab} is not 0/1"
        out.append(dict(id=str(cid), text=str(text), label=lab))
    return sorted(out, key=lambda d: d["id"])

DOCS = {"train": parse_cases(read_json(get_path("Train/train.json")), True),
        "dev": parse_cases(read_json(get_path("Train/dev.json")), True),
        "test": parse_cases(read_json(get_path("Test/test.json")), False)}
def dedupe_train(ds, label="train"):
    """The train split merges the single- and multi-instance data, so ids can repeat. Exact copies are dropped; different documents sharing an id get a suffix (train ids are never submitted)."""
    groups, out, dropped, renamed = {}, [], 0, 0
    for d in ds:
        g = groups.setdefault(d["id"], [])
        if any(o["text"] == d["text"] and o["label"] == d["label"] for o in g): dropped += 1; continue
        if g: d = dict(d, id=f'{d["id"]}__{len(g)}'); renamed += 1
        g.append(d); out.append(d)
    if dropped or renamed: print(f"{label}: {dropped} exact duplicate cases dropped, {renamed} cases with a repeated id renamed")
    return sorted(out, key=lambda d: d["id"])
for sp, ds in DOCS.items():
    assert sum(d["id"] == "None" for d in ds) < max(1, len(ds) // 2), f"{sp}: most cases have no id field - check the json keys (id / case_id)"
    print(f"{sp}: sample ids {[d['id'] for d in ds[:3]]}")
DOCS["train"] = dedupe_train(DOCS["train"], "train"); DOCS["dev"] = dedupe_train(DOCS["dev"], "dev")
n_dup_test = len(DOCS["test"]) - len({d["id"] for d in DOCS["test"]})
if n_dup_test: print(f"WARNING: {n_dup_test} repeated ids in test.json - kept as they are (one prediction per test entry, same order)")
if TRAIN_DOC_LIMIT and len(DOCS["train"]) > TRAIN_DOC_LIMIT: DOCS["train"] = random.Random(SEED).sample(DOCS["train"], TRAIN_DOC_LIMIT)
if SMOKE:
    DOCS = {"train": DOCS["train"][:16], "dev": DOCS["dev"][:8], "test": DOCS["test"][:6]}; EPOCHS = {k: 1 for k in EPOCHS}; K_CHUNKS = 2
    print(">>> SMOKE MODE: tiny subset, 1 epoch, 2 chunks <<<")
for sp, ds in DOCS.items():
    wl = [len(d["text"].split()) for d in ds]
    lab = f" | class 0/1: {sum(d['label']==0 for d in ds)}/{sum(d['label']==1 for d in ds)}" if ds[0]["label"] is not None else ""
    print(f"{sp:5s}: {len(ds):6d} cases | words/case median {int(np.median(wl))}, max {max(wl)}{lab}")

def pick_chunks(nc, K, mode):
    if nc <= K: return list(range(nc))
    if mode == "tail": return list(range(nc - K, nc))
    if mode == "head": return list(range(K))
    if mode == "headtail": h = K // 2; return list(range(h)) + list(range(nc - (K - h), nc))
    if mode == "spread": return sorted(set(np.linspace(0, nc - 1, K).round().astype(int).tolist()))
    raise ValueError(mode)

def pack_docs(docs, wt, mode, K):
    N = len(docs); X = np.full((N, K, WINDOW), wt.pad, np.int16); M = np.zeros((N, K), bool); P = np.zeros((N, K), np.float32)
    y = np.array([d["label"] if d["label"] is not None else -1 for d in docs], np.int64)
    for i, d in enumerate(docs):
        words = d["text"].split(); nc = max(1, -(-len(words) // CHUNK_WORDS))
        for k, c in enumerate(pick_chunks(nc, K, mode)):
            ids = wt.encode_words(words[c * CHUNK_WORDS:(c + 1) * CHUNK_WORDS], WINDOW)
            X[i, k, :len(ids)] = ids; M[i, k] = True; P[i, k] = c / max(nc - 1, 1)
    return dict(X=torch.from_numpy(X), M=torch.from_numpy(M), P=torch.from_numpy(P), y=torch.from_numpy(y))

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

train: sample ids ['1947_1', '1947_100', '1947_101']
dev: sample ids ['1950_22', '1950_22', '1950_27']
test: sample ids ['1950_14', '1950_23', '1950_38']
train: 5082 exact duplicate cases dropped, 0 cases with a repeated id renamed
dev: 994 exact duplicate cases dropped, 0 cases with a repeated id renamed
train:  15000 cases | words/case median 2040, max 412606 | class 0/1: 8869/6131
dev  :    994 cases | words/case median 2821, max 42198 | class 0/1: 497/497
test :   1517 cases | words/case median 2940, max 40487


In [7]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

class DocModel(nn.Module):
    """Phase-1 backbone -> mean-pool tokens per chunk -> (attention) pool chunks per document -> 2 logits."""
    def __init__(self, arch, backbone, d_model, pad_id, bidirectional=True, pool="attn", use_pos=True):
        super().__init__()
        self.arch, self.backbone, self.bidir, self.pool, self.use_pos, self.pad_id, self.d_model = arch, backbone, bidirectional, pool, use_pos, pad_id, d_model
        self.att = nn.Linear(d_model + (1 if use_pos else 0), 1)
        self.fmean, self.fstd = nn.Parameter(torch.zeros(d_model), requires_grad=False), nn.Parameter(torch.ones(d_model), requires_grad=False); self._collect = None
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(d_model, 256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, 2))
        if bidirectional and arch in CAUSAL_ARCHS:
            for mod in backbone.modules():
                if hasattr(mod, "causal_mask"): mod.causal_mask.fill_(True)

    def token_features(self, ids):
        m = self.backbone; e = m.token_emb(ids); T = e.shape[1]
        if self.arch == "decoder":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0))
            for b in m.blocks: x = b(x)
            return m.ln_f(x)
        if self.arch == "encdec":
            x = m.drop(e + m.enc_pos[:T].unsqueeze(0))
            for b in m.encoder_blocks: x = b(x)
            return m.enc_ln_f(x)
        if self.arch == "hybrid":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0)); plen = T if self.bidir else 0
            for b in m.blocks: x = b(x, plen)
            return m.ln_f(x)
        x = m.drop(e)
        for b in m.blocks: x = b(x, m.rope_cos, m.rope_sin)
        return m.norm_f(x)

    def forward(self, X, M, P):
        B, K, T = X.shape; ids = X.reshape(B * K, T).long(); valid = M.reshape(-1)
        h = self.token_features(ids[valid]).float()
        tm = (ids[valid] != self.pad_id).float().unsqueeze(-1)
        vec = (h * tm).sum(1) / tm.sum(1).clamp(min=1)
        if self._collect is not None: self._collect.append(vec.detach().float().cpu())
        vec = (vec - self.fmean) / self.fstd
        full = torch.zeros(B * K, self.d_model, device=vec.device, dtype=vec.dtype); full[valid] = vec; full = full.view(B, K, -1)
        if self.pool == "mean": w = M.float() / M.float().sum(1, keepdim=True).clamp(min=1)
        else:
            s = self.att(torch.cat([full, P.unsqueeze(-1).to(full.dtype)], -1) if self.use_pos else full).squeeze(-1)
            w = torch.softmax(s.float().masked_fill(~M, -1e9), 1)
        return self.head((w.unsqueeze(-1) * full).sum(1).to(self.head[1].weight.dtype))

In [8]:
class Adapter(nn.Module):
    """Houlsby bottleneck adapter; up-projection starts at zero so the model starts identical to Phase 1."""
    def __init__(self, d, bottleneck):
        super().__init__(); self.down, self.up, self.act = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.GELU()
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(self.act(self.down(x)))

class Adapted(nn.Module):
    def __init__(self, inner, d, bottleneck): super().__init__(); self.inner, self.adapter = inner, Adapter(d, bottleneck)
    def forward(self, *a, **k): return self.adapter(self.inner(*a, **k))

def _swap(root, names, make):
    n = 0
    for _, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(mod, cname, make(child)); n += 1
    return n

def apply_method(model, method):
    bb, P = model.backbone, PEFT
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "lora": n = _swap(bb, P["targets"], lambda b: LoRALinear(b, P["r"], P["alpha"]))
        elif method == "lorma": n = _swap(bb, P["lorma_targets"], lambda b: LoRMALinear(b, P["r"], P["alpha"]))
        elif method == "adapter":
            n = 0
            for blk in getattr(bb, block_list_name(model.arch)):
                for cname, child in list(blk.named_children()):
                    if cname in ("attn", "self_attn", "ff"): setattr(blk, cname, Adapted(child, model.d_model, P["bottleneck"])); n += 1
        elif method == "bitfit":
            n = 0
            for nm, p in bb.named_parameters():
                if nm.endswith(".bias"): p.requires_grad = True; n += 1
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                    for p in mod.parameters(): p.requires_grad = True; n += 1
        elif method == "partial":
            blocks = list(getattr(bb, block_list_name(model.arch))); n = 0
            for blk in blocks[-P["partial_last"]:]:
                for p in blk.parameters(): p.requires_grad = True; n += 1
            for nm, p in bb.named_parameters():
                if nm.split(".")[0] in ("ln_f", "norm_f", "enc_ln_f"): p.requires_grad = True; n += 1
        else: raise ValueError(f"unknown method {method}")
        assert n > 0, f"{method}: nothing to adapt"
    for p in list(model.head.parameters()) + list(model.att.parameters()): p.requires_grad = True
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad); tot = sum(p.numel() for p in model.parameters())
    print(f"[{method}] trainable {tr/1e6:.3f}M of {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad} | {"fmean", "fstd"}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

In [9]:
def macro_prf(y, p):
    cm = np.bincount(y * 2 + p, minlength=4).reshape(2, 2); tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    pr = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0); rc = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    return float(f1.mean()), float(pr.mean()), float(rc.mean())

def tune_threshold(score, y, grid=np.arange(-3, 3.01, 0.1), min_gain=0.002):
    base = macro_prf(y, (score > 0).astype(np.int64))[0]; best, bt = base, 0.0
    for t in grid:
        s = macro_prf(y, (score > t).astype(np.int64))[0]
        if s > best + 1e-9 or (abs(s - best) <= 1e-9 and abs(t) < abs(bt)): best, bt = s, float(t)
    return (bt, best) if best - base >= min_gain else (0.0, base)

def build_doc(arch, p1_path, method, keep_every=None):
    bb, cfg, max_len = load_phase1_backbone(arch, p1_path); strip_unused(bb, arch)
    if keep_every: prune_layers(bb, arch, list(range(0, len(getattr(bb, block_list_name(arch))), keep_every)))
    assert WINDOW <= max_len, f"WINDOW {WINDOW} exceeds the Phase-1 max length {max_len}"
    model = DocModel(arch, bb, cfg["d_model"], PAD_ID, BIDIRECTIONAL, POOL, USE_POS_FEATURE); apply_method(model, method)
    return model.to(DEVICE)

@torch.no_grad()
def predict_logits(model, pk, bs=4):
    model.eval(); out = []
    for b in range(0, len(pk["X"]), bs):
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            lg = model(pk["X"][b:b + bs].to(DEVICE), pk["M"][b:b + bs].to(DEVICE), pk["P"][b:b + bs].to(DEVICE))
        out.append(lg.float().cpu())
    return torch.cat(out).numpy()

def loss_fn(lg, y, w, t_lg=None):
    hard = F.cross_entropy(lg.float(), y, weight=w, label_smoothing=LABEL_SMOOTH)
    if t_lg is None: return hard
    soft = F.kl_div(F.log_softmax(lg.float() / KD_T, -1), F.softmax(t_lg.float() / KD_T, -1), reduction="batchmean") * KD_T * KD_T
    return KD_ALPHA * hard + (1 - KD_ALPHA) * soft

def fit(model, pk, eval_fn, ckdir, method, class_w, teacher=None, seed=0):
    os.makedirs(ckdir, exist_ok=True); last_p, best_p = f"{ckdir}/last.pt", f"{ckdir}/best.pt"
    dm = nn.DataParallel(model) if NUM_GPUS > 1 else model
    small = {id(p) for p in list(model.head.parameters()) + list(model.att.parameters())}
    tr = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": [p for p in tr if id(p) in small], "lr": HEAD_LR}, {"params": [p for p in tr if id(p) not in small], "lr": LR[method]}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
    base = [g["lr"] for g in opt.param_groups]; scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    ds = torch.utils.data.TensorDataset(pk["X"], pk["M"], pk["P"], pk["y"])
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_DOCS, shuffle=True, drop_last=len(ds) > BATCH_DOCS, generator=torch.Generator().manual_seed(seed))
    epochs = EPOCHS[method]; spe = max(len(loader) // GRAD_ACCUM, 1); total, warm = epochs * spe, max(int(WARMUP_FRAC * epochs * spe), 1)
    cw = class_w.to(DEVICE); start, best, hist, bad = 0, -1.0, [], 0
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best, hist, bad = ck["epoch"], ck["best"], ck["hist"], ck["bad"]; print(f"resumed at epoch {start}/{epochs}")
    step, t0 = start * spe, time.time()
    for ep in range(start, epochs):
        dm.train(); run = nb = skipped = good = 0; opt.zero_grad(set_to_none=True)
        for i, (x, m, p, y) in enumerate(loader):
            x, m, p, y = x.to(DEVICE), m.to(DEVICE), p.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                lg = dm(x, m, p); tl = None
                if teacher is not None:
                    with torch.no_grad(): tl = teacher(x, m, p)
                loss = loss_fn(lg, y, cw, tl) / GRAD_ACCUM
            if torch.isfinite(loss.detach()): scaler.scale(loss).backward(); run += loss.item() * GRAD_ACCUM; nb += 1; good += 1
            else: skipped += 1
            if (i + 1) % GRAD_ACCUM == 0 and good:
                f = (step + 1) / warm if step < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))
                for g, b in zip(opt.param_groups, base): g["lr"] = b * f
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step % 25 == 0: print(f"    step {step}/{total} | running loss {run / max(nb, 1):.4f}", flush=True)
            if (i + 1) % GRAD_ACCUM == 0:
                if not good: opt.zero_grad(set_to_none=True)
                good = 0
        f1 = eval_fn(model); hist.append(dict(epoch=ep + 1, loss=run / max(nb, 1), dev_macro_f1=f1))
        print(f"  epoch {ep+1}/{epochs} | loss {run/max(nb,1):.4f} | dev macro-F1 {f1:.4f} | {(time.time()-t0)/60:.1f} min" + getattr(eval_fn, "stats", "") + (f" | skipped {skipped} non-finite batches" if skipped else ""), flush=True)
        if f1 > best: best, bad = f1, 0; torch.save({"trainable": trainable_state(model)}, best_p + ".tmp"); os.replace(best_p + ".tmp", best_p)
        else: bad += 1
        torch.save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "epoch": ep + 1, "best": best, "hist": hist, "bad": bad}, last_p + ".tmp"); os.replace(last_p + ".tmp", last_p)
        if bad >= PATIENCE: print("  early stop (patience)"); break
        if RUN_MAX_MIN and (time.time() - t0) / 60 > RUN_MAX_MIN: print("  stopping: time limit for this run"); break
    model.load_state_dict(torch.load(best_p, map_location="cpu", weights_only=False)["trainable"], strict=False)
    return hist, best

In [ ]:
Y_DEV = np.array([d["label"] for d in DOCS["dev"]], dtype=np.int64)
def calibrate_chunk_words(wt, n=300):
    """Largest words-per-chunk such that ~85% of chunks still fit in WINDOW tokens (the old fixed 200 words overflowed 512 tokens and the start of every chunk was dropped)."""
    ds = random.Random(SEED).sample(DOCS["train"], min(n, len(DOCS["train"]))); r = []
    for d in ds:
        ws = d["text"].split()[:3000]
        if ws: r.append(len(wt.encode_words(ws, 10 ** 9)) / len(ws))
    tpw = float(np.percentile(r, 85)); cw = max(50, int(0.97 * WINDOW / tpw))
    print(f"tokenizer: {np.mean(r):.2f} tokens/word (p85 {tpw:.2f}) -> CHUNK_WORDS = {cw}  (was 200: {100 * float(np.mean([200 * x > WINDOW for x in r])):.0f}% of 200-word chunks would exceed {WINDOW} tokens)")
    return cw

def setup_arch(arch):
    global ARCH, WORK, P1_MODEL, wt, TOK_SHA, PAD_ID, PACKS, CHUNK_WORDS
    ARCH = arch; WORK = os.path.join(OUT_ROOT, f"{EXP_TAG}_{arch}"); os.makedirs(WORK, exist_ok=True); PACKS = {}
    P1_MODEL, p1_tok = find_phase1(arch); wt = WordTok(BPE.load(p1_tok)); PAD_ID = wt.pad
    if CHUNK_WORDS == "auto": CHUNK_WORDS = calibrate_chunk_words(wt)
    TOK_SHA = hashlib.sha256(open(p1_tok, "rb").read()).hexdigest(); print(f"\n##### {arch}: model {P1_MODEL} | tokenizer {TOK_SHA[:10]}")
def get_packs(mode, K):
    if (mode, K) not in PACKS: PACKS[(mode, K)] = {sp: pack_docs(DOCS[sp], wt, mode, K) for sp in DOCS}
    return PACKS[(mode, K)]

def write_submission(pred, name, outdir):
    sub = [{"id": d["id"], "pred": int(p)} for d, p in zip(DOCS["test"], pred)]
    pj = f"{outdir}/prediction.json"; json.dump(sub, open(pj, "w")); zp = f"{OUT_ROOT}/{name}_submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    return zp

@torch.no_grad()
def calibrate_features(model, pk, n=600):
    """Measure mean/std of the pooled chunk vectors on random training documents and standardise with them."""
    idx = torch.randperm(len(pk["X"]), generator=torch.Generator().manual_seed(0))[:n]
    model._collect = []; predict_logits(model, {k: v[idx] for k, v in pk.items()})
    v = torch.cat(model._collect); model._collect = None
    model.fmean.data.copy_(v.mean(0).to(model.fmean.device)); model.fstd.data.copy_(v.std(0).clamp(min=1e-2).to(model.fstd.device))
    print(f"  feature standardisation set from {len(v)} chunks (raw std across dims: median {v.std(0).median():.3f})")

def run_one(spec):
    method = spec["method"]; name = spec.get("name", method); mode = spec.get("mode", DOC_MODE); K = spec.get("k", K_CHUNKS); seed = int(spec.get("seed", 0))
    outdir = f"{WORK}/{name}"; os.makedirs(outdir, exist_ok=True)
    if os.path.exists(f"{outdir}/done.json") and json.load(open(f"{outdir}/done.json")).get("smoke") != SMOKE:
        print(f"[{ARCH}/{name}] found a leftover {'full' if SMOKE else 'smoke-test'} result -> deleting it and running fresh"); shutil.rmtree(outdir); os.makedirs(outdir, exist_ok=True)
    if os.path.exists(f"{outdir}/done.json"):
        print(f"[{ARCH}/{name}] already done -> skipped"); zp = f"{OUT_ROOT}/{EXP_TAG}_{ARCH}_{name}_submission.zip"
        if not os.path.exists(zp) and os.path.exists(f"{outdir}/prediction.json"):
            with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{outdir}/prediction.json", arcname="prediction.json")
        return json.load(open(f"{outdir}/done.json"))
    print(f"\n{'='*20} {ARCH} / {name}  (mode={mode}, K={K}) {'='*20}"); t_start = time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    random.seed(SEED + seed); np.random.seed(SEED + seed); torch.manual_seed(SEED + seed)
    model = build_doc(ARCH, P1_MODEL, method, STUDENT_KEEP_EVERY if method == "kd" else None); pk = get_packs(mode, K)
    cnt = np.bincount(pk["train"]["y"].numpy(), minlength=2).astype(np.float64)
    w = np.sqrt(cnt.max() / np.maximum(cnt, 1)) if CLASS_BALANCE == "sqrt" else np.ones(2)
    teacher = None
    if method == "kd":
        cands = [m for m in (json.load(open(p)) for p in glob.glob(f"{WORK}/*/done.json")) if m["method"] != "kd" and m["mode"] == mode and m["k"] == K]
        assert cands, "kd needs a finished non-kd run of this model (same mode/k) earlier in its list"
        tm = max(cands, key=lambda m: m["dev_f1_raw"]); print(f"  teacher = {tm['name']} (dev macro-F1 {tm['dev_f1_raw']:.4f}); student keeps every {STUDENT_KEEP_EVERY}nd block")
        teacher = build_doc(ARCH, P1_MODEL, tm["method"])
        teacher.load_state_dict(torch.load(f"{WORK}/{tm['name']}/ckpt/best.pt", map_location="cpu", weights_only=False)["trainable"], strict=False)
        teacher.eval(); [p.requires_grad_(False) for p in teacher.parameters()]
    def ev(m):
        l = predict_logits(m, pk["dev"]); s = l[:, 1] - l[:, 0]
        ev.stats = f" | dev predicts class 1 for {100 * float((s > 0).mean()):.0f}%, score std {float(s.std()):.2f}"
        return tune_threshold(s, Y_DEV)[1] if THRESH_TUNE else macro_prf(Y_DEV, (s > 0).astype(np.int64))[0]
    if FEAT_STD: calibrate_features(model, pk["train"])
    hist, best = fit(model, pk["train"], ev, f"{outdir}/ckpt", method, torch.tensor(w, dtype=torch.float32), teacher, seed=seed)
    _ti = time.time(); ld = predict_logits(model, pk["dev"]); infer_ms = 1000 * (time.time() - _ti) / len(Y_DEV); lt = predict_logits(model, pk["test"]); np.save(f"{outdir}/lg_dev.npy", ld); np.save(f"{outdir}/lg_test.npy", lt)
    sd, st = ld[:, 1] - ld[:, 0], lt[:, 1] - lt[:, 0]; raw = macro_prf(Y_DEV, (sd > 0).astype(np.int64))
    th, tuned = tune_threshold(sd, Y_DEV) if THRESH_TUNE else (0.0, raw[0])
    cm = np.bincount(Y_DEV * 2 + (sd > th).astype(np.int64), minlength=4).tolist()
    meta = dict(task="phase4_cjpe", name=name, arch=ARCH, method=method, mode=mode, k=K, tag=EXP_TAG, tokenizer_sha=TOK_SHA, doc_ids_dev=[d["id"] for d in DOCS["dev"]],
                doc_ids_test=[d["id"] for d in DOCS["test"]], dev_f1_raw=raw[0], dev_precision=raw[1], dev_recall=raw[2], dev_f1_tuned=tuned, threshold=th, lr=LR[method],
                trainable_M=round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3), total_M=round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
                seed=seed, chunk_words=CHUNK_WORDS, dev_confusion=cm, train_class_counts=cnt.tolist(), infer_ms_per_doc=round(infer_ms, 1), peak_mem_GB=round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else 0.0, epochs_run=len(hist), minutes=round((time.time() - t_start) / 60, 1), smoke=SMOKE)
    json.dump(meta, open(f"{outdir}/meta.json", "w"), indent=1)
    zp = write_submission((st > th).astype(np.int64), f"{EXP_TAG}_{ARCH}_{name}", outdir)
    print(f"  dev macro-F1: raw {raw[0]:.4f} (P {raw[1]:.3f} R {raw[2]:.3f}) -> threshold-tuned {tuned:.4f}\n  submission: {zp}")
    json.dump(meta, open(f"{outdir}/done.json", "w"))
    lp = f"{outdir}/ckpt/last.pt"
    if os.path.exists(lp): os.remove(lp)
    del model, teacher; gc.collect(); torch.cuda.empty_cache()
    return meta

def zip_dir(zpath, files):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
        for f, arc in files:
            if os.path.exists(f): z.write(f, arc)
    print(f"{os.path.getsize(zpath)/1e6:.1f} MB  {zpath}")

RESULTS = []
for arch, runs in PLAN:
    setup_arch(arch)
    for s in runs: RESULTS.append(run_one(s))
    done = [os.path.dirname(p) for p in glob.glob(f"{WORK}/*/done.json")]
    zip_dir(f"{OUT_ROOT}/{EXP_TAG}_{arch}_bundle.zip", [(f"{d}/{n}", f"{os.path.basename(WORK)}/{os.path.basename(d)}/{n}") for d in done for n in ("lg_dev.npy", "lg_test.npy", "meta.json")])
print("\nSESSION SUMMARY"); [print(f"  {m['arch']:11s} {m['name']:8s} raw {m['dev_f1_raw']:.4f} | tuned {m['dev_f1_tuned']:.4f} | {m['minutes']} min") for m in RESULTS]
print("Plan finished. The ensemble and the final zip come next (next cells).")


tokenizer: 2.21 tokens/word (p85 2.27) -> CHUNK_WORDS = 218  (was 200: 0% of 200-word chunks would exceed 512 tokens)

##### hybrid: model /kaggle/input/datasets/sparkles1234/hybrid-240059/hybrid_model.pt | tokenizer 8623378cb3

==================== hybrid / full  (mode=tail, K=4) ====================
[full] trainable 108.604M of 108.60M (100.00%)
  feature standardisation set from 2167 chunks (raw std across dims: median 0.265)
    step 25/3750 | running loss 0.7063
    step 50/3750 | running loss 0.7602
    step 75/3750 | running loss 0.7965
    step 100/3750 | running loss 0.7830
    step 125/3750 | running loss 0.7734
    step 150/3750 | running loss 0.7701
    step 175/3750 | running loss 0.7756
    step 200/3750 | running loss 0.7897
    step 225/3750 | running loss 0.7813
    step 250/3750 | running loss 0.7951
    step 275/3750 | running loss 0.7948
    step 300/3750 | running loss 0.7898
    step 325/3750 | running loss 0.7835
    step 350/3750 | running loss 0.7770
    step 3

In [ ]:

def run_ensemble():
    metas = []
    for root in (OUT_ROOT, INPUT_ROOT):
        for mp in glob.glob(os.path.join(root, "**", "meta.json"), recursive=True):
            m = json.load(open(mp))
            d = os.path.dirname(mp)
            if m.get("task") == "phase4_cjpe" and not m.get("smoke") and os.path.exists(os.path.join(d, "lg_dev.npy")):
                m["_dir"], m["_t"] = d, os.path.getmtime(os.path.join(d, "lg_dev.npy")); metas.append(m)
    seen = {}
    for m in sorted(metas, key=lambda m: -m["_t"]): seen.setdefault((m["arch"], m["name"]), m)
    metas = sorted(seen.values(), key=lambda m: -m["dev_f1_raw"])
    if not metas: print("no finished (non-smoke) runs found - nothing to ensemble"); return
    metas = [m for m in metas if m["doc_ids_dev"] == metas[0]["doc_ids_dev"] and m["doc_ids_test"] == metas[0]["doc_ids_test"]]
    prob1 = lambda l: (lambda e: (e / e.sum(1, keepdims=True))[:, 1])(np.exp(l - l.max(1, keepdims=True)))
    logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1))
    groups = {}
    for m in metas: groups.setdefault(m["arch"] if ENSEMBLE_SAME_ARCH else "allarch", []).append(m)
    for gname, ms in groups.items():
        PD = [prob1(np.load(m["_dir"] + "/lg_dev.npy")) for m in ms]; PT = [prob1(np.load(m["_dir"] + "/lg_test.npy")) for m in ms]
        base = {}; [base.__setitem__(m["arch"], max(base.get(m["arch"], 0), m["total_M"])) for m in ms]
        est = sum(base.values()) + sum(m["trainable_M"] for m in ms)
        print(f"\n[{gname}] {len(ms)} runs, approx. ensemble size {est:.0f}M params (limit {MAX_PARAMS_M}M)" + ("  <-- OVER the limit: use fewer runs" if est > MAX_PARAMS_M else ""))
        for m in ms: print(f"  {m['arch']:11s} {m['name']:8s} dev macro-F1 {m['dev_f1_raw']:.4f}")
        def finish(idx, name):
            sd = logit(np.mean([PD[i] for i in idx], 0)); st = logit(np.mean([PT[i] for i in idx], 0)); raw = macro_prf(Y_DEV, (sd > 0).astype(np.int64))[0]
            th, tuned = tune_threshold(sd, Y_DEV) if THRESH_TUNE else (0.0, raw)
            tag = f"{gname}_{name}"; od = f"{OUT_ROOT}/{EXP_TAG}_ensemble_{tag}"; os.makedirs(od, exist_ok=True)
            zp = write_submission((st > th).astype(np.int64), f"{EXP_TAG}_ensemble_{tag}", od)
            print(f"  {name:8s} members={len(idx)} dev macro-F1 raw {raw:.4f} | threshold-tuned {tuned:.4f} -> {os.path.basename(zp)}")
        finish(list(range(len(ms))), "all"); finish(list(range(min(4, len(ms)))), "top4")
        sel, best = [0], macro_prf(Y_DEV, (PD[0] > 0.5).astype(np.int64))[0]
        while True:
            gains = [(macro_prf(Y_DEV, (np.mean([PD[j] for j in sel + [i]], 0) > 0.5).astype(np.int64))[0], i) for i in range(len(ms)) if i not in sel]
            if not gains or max(gains)[0] <= best + 1e-4: break
            best, nxt = max(gains); sel.append(nxt)
        finish(sel, "greedy")

if ENSEMBLE:
    run_ensemble()


In [ ]:

import csv
rows = []
for p in sorted(glob.glob(f"{OUT_ROOT}/{EXP_TAG}_*/*/done.json")):
    rows.append(json.load(open(p)))
cols = ["arch", "name", "method", "mode", "k", "dev_f1_raw", "dev_precision", "dev_recall", "dev_f1_tuned", "threshold", "trainable_M", "total_M", "epochs_run", "minutes", "infer_ms_per_doc", "peak_mem_GB", "dev_confusion", "train_class_counts", "seed", "chunk_words"]
summary = f"{OUT_ROOT}/{EXP_TAG}_results_summary.csv"
with open(summary, "w", newline="") as f:
    w = csv.writer(f); w.writerow(cols); [w.writerow([r.get(c, "") for c in cols]) for r in rows]
final_zip = f"{OUT_ROOT}/{EXP_TAG}_ALL_RESULTS.zip"
files = []
for name in sorted(os.listdir(OUT_ROOT)):
    p = os.path.join(OUT_ROOT, name)
    if p == final_zip or not name.startswith(f"{EXP_TAG}_"): continue
    if os.path.isdir(p):
        files += [(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), OUT_ROOT)) for dp, _, fs in os.walk(p) for f in fs if f != "last.pt" and not f.endswith(".tmp")]
    else: files.append((p, name))
zip_dir(final_zip, files)
print(f"{len(files)} files saved. Table of all runs:"); [print(f"  {r['arch']:11s} {r['name']:8s} {r['method']:8s} raw {r['dev_f1_raw']:.4f} | tuned {r['dev_f1_tuned']:.4f}") for r in rows]
print("\nDownload", final_zip, "from the Output panel BEFORE closing (single-run and ensemble *_submission.zip files are inside it and also in the Output panel).")
